# 02 — Red Bayesiana: Riesgo de Roya del Café

Este notebook demuestra el módulo `src/bayesian/disease_risk.py`.

**Objetivo:** estimar el riesgo de *Hemileia vastatrix* (roya del café) **antes de que aparezcan síntomas visibles**, usando únicamente variables climáticas observables.

### Estructura causal
```
Lluvia → Humedad → RiesgoRoya ← Temperatura
                        ↓
               RendimientoEsperado
```

### Variables
| Variable | Estados |
|---|---|
| Temperatura | 0=Fresca (<18°C), 1=Óptima (18-24°C), 2=Caliente (>24°C) |
| Humedad | 0=Baja (<70%), 1=Alta (≥70%) |
| Lluvia | 0=Poca, 1=Moderada, 2=Intensa |
| RiesgoRoya | 0=Bajo, 1=Alto |
| RendimientoEsperado | 0=Bajo, 1=Medio, 2=Alto |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.bayesian.disease_risk import evaluate_disease_risk, get_yield_forecast, _model

print("Modelo cargado. Nodos:", list(_model.nodes()))
print("Aristas:", list(_model.edges()))

## 1. Visualización de la estructura de la red

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

G = nx.DiGraph(_model.edges())

pos = {
    "Lluvia":              (-1.5,  1.0),
    "Temperatura":         ( 1.5,  1.0),
    "Humedad":             (-0.5,  0.0),
    "RiesgoRoya":          ( 0.5,  0.0),
    "RendimientoEsperado": ( 0.5, -1.0),
}

colores = {
    "Lluvia":              "#4fa3d1",
    "Temperatura":         "#e07b4f",
    "Humedad":             "#6abf69",
    "RiesgoRoya":          "#d9534f",
    "RendimientoEsperado": "#f0ad4e",
}

fig, ax = plt.subplots(figsize=(8, 5))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_color=[colores[n] for n in G.nodes()],
    node_size=2800,
    font_size=10, font_color="white", font_weight="bold",
    edge_color="#555", arrows=True, arrowsize=20,
    connectionstyle="arc3,rad=0.05",
)
ax.set_title("Red Bayesiana — Riesgo de Roya del Café", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Consultas puntuales — evaluate_disease_risk()

In [ ]:
import pandas as pd

escenarios = [
    (1, 1, 2, "Óptima + humedad alta + lluvia intensa  ← peor caso"),
    (1, 1, 1, "Óptima + humedad alta + lluvia moderada"),
    (1, 0, 0, "Óptima + humedad baja + poca lluvia"),
    (0, 1, 2, "Fresca  + humedad alta + lluvia intensa"),
    (2, 0, 0, "Caliente + humedad baja + poca lluvia   ← mejor caso"),
]

filas = []
for temp, hum, lluvia, desc in escenarios:
    r = evaluate_disease_risk(temp, hum, lluvia)
    y = get_yield_forecast(temp, hum, lluvia)
    filas.append({
        "Escenario":          desc,
        "P(riesgo bajo)":     round(r["riesgo_bajo"],  3),
        "P(riesgo alto)":     round(r["riesgo_alto"],  3),
        "Nivel riesgo":       r["nivel"],
        "Rendimiento esp.":   y["nivel_esperado"],
    })

df = pd.DataFrame(filas)
df.style.applymap(
    lambda v: "background-color: #f9c0c0" if v == "ALTO" else
              "background-color: #c0f0c0" if v == "BAJO" else "",
    subset=["Nivel riesgo"]
)

## 3. Tabla de sensibilidad completa — los 18 escenarios

In [ ]:
labels_t = {0: "Fresca",   1: "Óptima",   2: "Caliente"}
labels_h = {0: "Baja",    1: "Alta"}
labels_l = {0: "Poca",    1: "Moderada", 2: "Intensa"}

filas_sens = []
for t in range(3):
    for h in range(2):
        for l in range(3):
            r = evaluate_disease_risk(t, h, l)
            filas_sens.append({
                "Temperatura": labels_t[t],
                "Humedad":     labels_h[h],
                "Lluvia":      labels_l[l],
                "P(riesgo alto)": round(r["riesgo_alto"], 3),
                "Nivel":          r["nivel"],
            })

df_sens = pd.DataFrame(filas_sens).sort_values("P(riesgo alto)", ascending=False)
df_sens.style.background_gradient(subset=["P(riesgo alto)"], cmap="RdYlGn_r")

## 4. Heatmap de riesgo: Temperatura × Humedad (fijando Lluvia = Intensa)

In [ ]:
import numpy as np
import seaborn as sns

matriz = np.zeros((3, 2))   # filas=Temperatura, columnas=Humedad
for t in range(3):
    for h in range(2):
        r = evaluate_disease_risk(t, h, lluvia=2)   # lluvia intensa
        matriz[t, h] = round(r["riesgo_alto"], 3)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    matriz,
    annot=True, fmt=".2f", cmap="RdYlGn_r",
    vmin=0, vmax=1,
    xticklabels=["Baja (<70%)", "Alta (≥70%)"],
    yticklabels=["Fresca (<18°C)", "Óptima (18-24°C)", "Caliente (>24°C)"],
    ax=ax, linewidths=0.5,
)
ax.set_xlabel("Humedad", fontsize=11)
ax.set_ylabel("Temperatura", fontsize=11)
ax.set_title("P(RiesgoRoya = Alto) | Lluvia Intensa", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Distribución de rendimiento esperado por escenario

In [ ]:
etiquetas = ["Ópt+Alt+Int\n(peor caso)", "Ópt+Alt+Mod", "Ópt+Baj+Poc", "Fres+Alt+Int", "Cal+Baj+Poc\n(mejor caso)"]
combos    = [(1,1,2), (1,1,1), (1,0,0), (0,1,2), (2,0,0)]

datos_bajo  = [get_yield_forecast(*c)["rendimiento_bajo"]  for c in combos]
datos_medio = [get_yield_forecast(*c)["rendimiento_medio"] for c in combos]
datos_alto  = [get_yield_forecast(*c)["rendimiento_alto"]  for c in combos]

x = np.arange(len(etiquetas))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, datos_bajo,  w, label="Rendimiento bajo",  color="#d9534f", alpha=0.85)
ax.bar(x,     datos_medio, w, label="Rendimiento medio", color="#f0ad4e", alpha=0.85)
ax.bar(x + w, datos_alto,  w, label="Rendimiento alto",  color="#5cb85c", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(etiquetas, fontsize=9)
ax.set_ylabel("Probabilidad")
ax.set_title("Distribución de RendimientoEsperado por escenario climático")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()